# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal checks first.**

Signal A — staleness (behind FlyRank's real `stale_visible_page` flag): pages that haven't
been updated in a long time are more likely to be declining.

Signal B — CTR vs position (behind FlyRank's real `low_ctr_visible_page` flag): CTR drops as
position gets worse.

**My rule, in plain words:** a page is worth a CTR fix if it gets meaningful search traffic,
sits at a position where users could reasonably find it (top 20), and its CTR is clearly below
what that position should deliver. Pages that are also stale get a small score boost as a
tiebreaker.

**Reason code:** `ctr_position_gap` (only code used; unflagged rows get none).
**Action label:** `refresh_and_review_ctr`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

# --- Signal A: freshness_tier vs decline rate ---
visible = df[df["impressions_90d"] >= 100].copy()
order_fresh = ["0-30", "31-90", "91-180", "181+"]
visible["freshness_tier"] = pd.Categorical(visible["freshness_tier"], categories=order_fresh, ordered=True)

signal_a = visible.groupby("freshness_tier", observed=True).agg(
    n=("content_id", "size"),
    decline_rate=("trend_direction", lambda s: round((s == "down").mean(), 4)),
    median_ctr=("ctr", "median"),
).round(4)
print("Signal A: freshness_tier vs decline_rate (visible pages, impressions_90d >= 100)")
print(signal_a)

# --- Signal B: position_tier vs CTR ---
order_pos = ["top_3", "page_1", "striking", "page_3_5", "deep"]
vis_pos = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
vis_pos["position_tier"] = pd.Categorical(vis_pos["position_tier"], categories=order_pos, ordered=True)

def weighted_ctr(g):
    return pd.Series({
        "n": len(g),
        "weighted_ctr_pct": round(100 * g["clicks_90d"].sum() / g["impressions_90d"].sum(), 3),
        "median_row_ctr": round(g["ctr"].median(), 3),
    })

signal_b = vis_pos.groupby("position_tier", observed=True).apply(weighted_ctr)
print("\nSignal B: position_tier vs CTR (visible pages, avg_position > 0)")
print(signal_b)

# bonus sanity check, not used as a rule input
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"\ncorr(search_volume, impressions_90d) = {corr:.4f} -> FALSE, excluded from rule")

**Verdict A: MIXED.** Decline rate rises with staleness in the two buckets that clear the
n-floor (`0-30`: n=13,735, decline_rate≈0.583 → `91-180`: n=8,084, decline_rate≈0.623), a real
but small ~4pt gap. `31-90` (n=152) and `181+` (n=35) don't line up cleanly, and `181+` is below
the ~50-row floor. Staleness alone is weak — real, small, not something to lean on by itself.

**Verdict B: CONFIRMED.** Weighted CTR falls monotonically: 0.487% (`top_3`, n=533) → 0.350%
(`page_1`, n=8,633) → 0.155% (`page_3_5`, n=6,058) → 0.039% (`deep`, n=879). Every bucket clears
the n-floor easily. Strongest, cleanest relationship in the dataset.

Inputs used in the rule below: `impressions_90d`, `avg_position`, `ctr`, `freshness_tier` — all
observable pre-decision signals, none derived from `trend_direction`/`trend_pct` (the label
source), no forward-looking window.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
visible = df["impressions_90d"] >= 100
findable = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
low_ctr = df["ctr"] < 0.5
ctr_position_gap = visible & findable & low_ctr
stale_tiebreak = df["freshness_tier"].isin(["91-180", "181+"])

df["reason_code"] = np.where(ctr_position_gap, "ctr_position_gap", "")
df["action_label"] = np.where(ctr_position_gap, "refresh_and_review_ctr", "no_action")
df["action_score"] = np.where(
    ctr_position_gap,
    np.log1p(df["impressions_90d"]) * (1 + 0.5 * stale_tiebreak.astype(int)),
    0.0,
).round(4)

n_flagged = int(ctr_position_gap.sum())
print(f"Flagged rows: {n_flagged} of {len(df)} ({n_flagged/len(df):.1%})")

queue_cols = ["content_id", "client_id", "impressions_90d", "avg_position", "ctr",
              "freshness_tier", "reason_code", "action_label", "action_score"]
queue = df.loc[ctr_position_gap, queue_cols].sort_values("action_score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_dir / "baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} ranked rows to work/outputs/baseline_action_score.csv")
queue.head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
for i, row in queue.head(10).iterrows():
    print(f"#{row['rank']}  {row['content_id']}  score={row['action_score']:.2f}")
    print(f"  action: {row['action_label']}  (reason: {row['reason_code']})")
    print(f"  why: impressions_90d={row['impressions_90d']:,}, avg_position={row['avg_position']:.1f}, "
          f"ctr={row['ctr']:.2f}%, freshness_tier={row['freshness_tier']}")
    print()

1. **#1** — Large, findable page (top-20, 500k+ impressions), CTR well under 0.5%. Wrong if
   the low CTR is a snippet-rendering issue, not a real content problem.
2. **#2** — Similar, also `down` trend. Wrong if the decline is client-wide, not page-specific.
3. **#3** — High volume, `stable`. Wrong if the query intent mismatch is structural.
4. **#4** — Similar profile. Wrong if this already ranks #1 for a commercial query where low
   CTR from snippet competition is normal — check `main_intent` first.
5. **#5** — High impressions, `down`. Wrong if impressions are about to fall regardless of CTR.
6. **#6** — `stable`, high volume. Wrong if it's a branded query with naturally lower CTR.
7. **#7** — High volume, `stable`. Wrong if the dip is seasonal.
8. **#8** — `down`. Wrong if `avg_position` averages many queries, hiding good performance on
   the main one.
9. **#9** — `stable`. Wrong if the page is already comprehensive and only the title needs work.
10. **#10** — `down`. Wrong if this is a temporary re-indexing artifact from a recent URL change.

**Pattern:** the rule surfaces high-impression, findable, chronically-low-CTR pages as
designed. Its blind spot: it can't distinguish a genuine fix opportunity from a page whose
lower CTR is legitimate for reasons outside its signals (intent, branding, seasonality).

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
score_inputs = {"impressions_90d", "avg_position", "ctr", "freshness_tier"}
label_source = {"trend_direction", "trend_pct", "is_declining_label"}
product_flags = {"health_score", "needs_ctr_fix", "is_quick_win", "priority_score", "action_type"}

assert score_inputs.isdisjoint(label_source), "leakage: label-source column used in score"
assert score_inputs.isdisjoint(product_flags), "leakage: product flag used in score"
assert product_flags.isdisjoint(df.columns), "product flags unexpectedly present in dataset"
print("Leakage check passed.")

**Weakest picks:** rows like #6/#7 — `stable` trend, flagged purely on the static CTR gap with
no urgency signal. Not wrong to include, but least differentiated from hundreds of similar rows
just outside the top 10; the rule has no strong way to break ties beyond the small stale-boost.

**Leakage check:** score inputs (`impressions_90d`, `avg_position`, `ctr`, `freshness_tier`) are
disjoint from label-source columns (`trend_direction`, `trend_pct`) and from FlyRank's product
flags, which aren't even in this dataset. `trend_direction` is read only for description in the
review above, never as a score input. No forward-looking window used anywhere.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.